# reranking with personalized pagerank

It turns out that global pagerank doesn't really help with reranking.
In this notebook, we explore reranking using personalized pagerank.
To do this, we need to read over all of the query results from the original dataset.
We then run personalized pagerank on the results of this.
This notebook is mostly for development.

In [1]:
%load_ext autoreload
%autoreload 2

In [33]:
from process import load_graph_data_remapped
from pathlib import Path
import polars as pl
import rustworkx as rx
from contexttimer import Timer

trec_root = Path("~/scratch/trec-tot-2025").expanduser()
dataset_root = trec_root / "data/enwiki/processed"
gdrive_path = trec_root / "data/gdrive/data"
official_path = trec_root / "data/official"
output_path = trec_root / "results/rerank"

In [22]:
p = dataset_root / "graph/v2/bge-m3-knn-k10/edges/*.parquet"
print(p)
edges = pl.read_parquet(p)
g, mapping_df = load_graph_data_remapped(edges)

/storage/home/hcoda1/8/amiyaguchi3/scratch/trec-tot-2025/data/enwiki/processed/graph/v2/bge-m3-knn-k10/edges/*.parquet


100%|██████████| 66142320/66142320 [00:42<00:00, 1553253.44it/s]


In [42]:
# nodes and edges
print("nodes", g.num_nodes())
print("edges", g.num_edges())

nodes 6614232
edges 66142320


In [10]:
cols = ["qid", "Q0", "docid", "rank", "score", "run_name"]
run = pl.read_csv(
    f"{gdrive_path}/shared_retrieval_results/gemini-2.5-flash/dev3.run",
    separator="\t",
    has_header=False,
    new_columns=cols,
)
# schema
display(run.collect_schema())
display(run.head(5))

Schema([('qid', Int64),
        ('Q0', String),
        ('docid', Int64),
        ('rank', Int64),
        ('score', Float64),
        ('run_name', String)])

qid,Q0,docid,rank,score,run_name
i64,str,i64,i64,f64,str
2001,"""Q0""",1752781,0,5.0,"""gemini-25_alias"""
2001,"""Q0""",58865,1,4.0,"""gemini-25_alias"""
2001,"""Q0""",4907816,2,4.0,"""gemini-25_alias"""
2001,"""Q0""",49363330,3,4.0,"""gemini-25_alias"""
2001,"""Q0""",73331355,4,4.0,"""gemini-25_alias"""


In [14]:
qid = 2001
subset = run.filter(pl.col("qid") == qid)
subset.shape

(12, 6)

In [28]:
# remap docid to idx in mapping df
mapped_subset = subset.select(pl.col("docid").alias("id")).join(
    mapping_df, on="id", how="inner"
)
mapped_subset

id,idx
i64,u32
58865,29272
1650010,458216
1752781,477811
4907816,961512
7561924,1258472
…,…
49363330,4719781
53500318,5018208
63153672,5715451


In [31]:
dim = mapped_subset.shape[0]
personalization = {idx: 1.0 / dim for idx in mapped_subset.select("idx").to_series()}
personalization

{29272: 0.08333333333333333,
 458216: 0.08333333333333333,
 477811: 0.08333333333333333,
 961512: 0.08333333333333333,
 1258472: 0.08333333333333333,
 3809180: 0.08333333333333333,
 4331587: 0.08333333333333333,
 4719781: 0.08333333333333333,
 5018208: 0.08333333333333333,
 5715451: 0.08333333333333333,
 5804836: 0.08333333333333333,
 6392395: 0.08333333333333333}

In [38]:
with Timer() as t:
    ppr = rx.pagerank(g, personalization=personalization, tol=1.0e-9)
print(f"elapsed {t} seconds")
len(ppr)

elapsed 34.775 seconds


6614232

In [39]:
pr_df = pl.DataFrame({"idx": ppr.keys(), "pagerank": ppr.values()}).join(
    mapped_subset, on="idx", how="inner"
)
pr_df

idx,pagerank,id
i64,f64,i64
29272,0.032399,58865
458216,0.014517,1650010
477811,0.014284,1752781
961512,0.020664,4907816
1258472,0.016398,7561924
…,…,…
4719781,0.015296,49363330
5018208,0.015901,53500318
5715451,0.014291,63153672


In [49]:
# and then write this result out as we get it
subset_reranked = (
    subset.drop("score")
    .join(
        pr_df.select(pl.col("id").alias("docid"), pl.col("pagerank").alias("score")),
        on="docid",
        how="inner",
    )
    .select(subset.columns)
).sort("score", descending=True)
subset_reranked

qid,Q0,docid,rank,score,run_name
i64,str,i64,i64,f64,str
2001,"""Q0""",58865,1,0.032399,"""gemini-25_alias"""
2001,"""Q0""",4907816,2,0.020664,"""gemini-25_alias"""
2001,"""Q0""",43936047,6,0.016869,"""gemini-25_alias"""
2001,"""Q0""",7561924,10,0.016398,"""gemini-25_alias"""
2001,"""Q0""",73331355,4,0.016064,"""gemini-25_alias"""
…,…,…,…,…,…
2001,"""Q0""",64323316,7,0.015528,"""gemini-25_alias"""
2001,"""Q0""",49363330,3,0.015296,"""gemini-25_alias"""
2001,"""Q0""",1650010,8,0.014517,"""gemini-25_alias"""
